# OSMI Mental Health in Tech (2016): Data Cleaning

This is a rework of my original Project 1 analysis. The first version used `WHERE IS NOT NULL` filters in SQL without actually checking what those nulls meant or how much data they represented. 
This notebook fixes that: I'm inspecting each issue in the raw data, deciding how to handle it, and 
documenting why.

Raw dataset: OSMI Mental Health in Tech 2016 survey, 1,433 responses, 63 columns.

In [1]:
import pandas as pd

df = pd.read_csv('mental-heath-in-tech-2016_20161114.csv')
print(df.shape)
df.columns.tolist()

(1433, 63)


['Are you self-employed?',
 'How many employees does your company or organization have?',
 'Is your employer primarily a tech company/organization?',
 'Is your primary role within your company related to tech/IT?',
 'Does your employer provide mental health benefits as part of healthcare coverage?',
 'Do you know the options for mental health care available under your employer-provided coverage?',
 'Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?',
 'Does your employer offer resources to learn more about mental health concerns and options for seeking help?',
 'Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?',
 'If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:',
 'Do you think that discussing a mental health disorder with your employer would have neg

## Missingness isn't random, it's survey branching logic

Before dropping or filling in anything, I checked *why* values were missing. This survey uses skip 
logic: if you say you're self-employed, you never see the employer-benefits questions. If you say 
you have no previous employers, you skip that whole block.

That means high missingness in a column isn't automatically a data quality problem. Some columns 
show 80-90% missing simply because most respondents' answers to an earlier question meant the 
follow-up didn't apply to them.

Decision: conditional columns like these are left untouched. NaN here means "not applicable," not 
"unknown." I only cleaned the columns that actually feed my four analysis queries (age, gender, 
remote work, treatment history, disorder status, supervisor comfort), where missingness was close 
to 0% and any gaps are real issues, not survey structure.

In [2]:
missing_pct = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
missing_pct.head(15)

If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?                                                        90.0
If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?                                                85.8
Is your primary role within your company related to tech/IT?                                                                                                                        81.6
Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?                                                               80.0
If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?                                                        80.0
If you have been diagnosed or treated for a mental health disorder, do you 

In [3]:
analysis_cols = [
    'What is your age?',
    'What is your gender?',
    'Do you work remotely?',
    'Have you ever sought treatment for a mental health issue from a mental health professional?',
    'Do you currently have a mental health disorder?',
    'Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?',
    'Do you think that discussing a mental health disorder with your employer would have negative consequences?',
]

df[analysis_cols].isna().mean().mul(100).round(1)

What is your age?                                                                                              0.0
What is your gender?                                                                                           0.2
Do you work remotely?                                                                                          0.0
Have you ever sought treatment for a mental health issue from a mental health professional?                    0.0
Do you currently have a mental health disorder?                                                                0.0
Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?                20.0
Do you think that discussing a mental health disorder with your employer would have negative consequences?    20.0
dtype: float64

In [4]:
df.groupby('Are you self-employed?')['Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?'].apply(lambda x: x.isna().mean() * 100)

Are you self-employed?
0      0.0
1    100.0
Name: Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?, dtype: float64

## Age: real outliers, not just extreme values

Age was self-reported, and a couple of respondents mistyped and/or misrepresented their age.

In [5]:
df['What is your age?'].describe()

count    1433.000000
mean       34.286113
std        11.290931
min         3.000000
25%        28.000000
50%        33.000000
75%        39.000000
max       323.000000
Name: What is your age?, dtype: float64

The max is 323 and the min is 3, neither is a real working-age respondent. Rather than picking an 
arbitrary cutoff, I checked how many rows actually fall outside a realistic working-age range 
(18-75) before deciding anything.

Only 5 rows out of 1,433 (0.3%) fell outside that range: ages 3, 15, 17, 99, and 323. That's too small a group to be worth guessing a replacement value for, so I dropped them.

In [6]:
age = df['What is your age?']
below_18 = age[age < 18]
above_75 = age[age > 75]

print("Below 18:", sorted(below_18.tolist()))
print("Above 75:", sorted(above_75.tolist()))
print("Total rows affected:", len(below_18) + len(above_75), "out of", len(df))

Below 18: [3, 15, 17]
Above 75: [99, 323]
Total rows affected: 5 out of 1433


## Gender: standardizing 71 free-text responses

Gender was also free text, producing 71 unique raw answers for what should be a handful of categories: capitalization and spacing variants ('Female', 'female ', 'F', 'f'), typos ('mail', 'Malr'), and a smaller set of non-binary, genderqueer, or unclear responses.

In [7]:
df['What is your gender?'].unique()

<StringArray>
[                                                                                                                                                         'Male',
                                                                                                                                                          'male',
                                                                                                                                                         'Male ',
                                                                                                                                                        'Female',
                                                                                                                                                             'M',
                                                                                                                                                        'female',
              

I split these into four categories instead of the usual two or three:
- **Male** / **Female**: standard variants, typos, and trans respondents classified by their 
  affirmed gender (e.g. "Transitioned, M2F" -> Female), not assumed birth sex 
- **Self-described / Other**: respondents who named an identity outside male/female (agender, genderfluid, 
  genderqueer, etc.), including single-word answers like "Other" that still tell you something (not male, not female) even without detail
- **Not reported**: blank answers, plus responses that give zero usable information rather than 
  naming an identity ("none of your business", "Human")

I kept "Not reported" separate from "Self-described / Other" on purpose. Lumping a real stated identity in with a refusal to answer blurs two very different things.

In [ ]:
import re

NONBINARY_KEYWORDS = [
    'agender', 'androgyn', 'bigender', 'enby', 'fluid', 'genderqueer',
    'nonbinary', 'non-binary', 'nb masculine', 'queer', 'unicorn', 'other', 'transfeminine',
]

NOT_REPORTED_KEYWORDS = [
    'none of your business', 'human'
]

def classify_gender(x):
    if pd.isna(x):
         return 'Not reported'
    s = str(x).strip().lower()

    for keyword in NOT_REPORTED_KEYWORDS:
        if keyword in s:
            return 'Not reported'

    for keyword in NONBINARY_KEYWORDS:
        if keyword in s:
            return 'Self-described / Other'

    if 'mtf' in s or 'm2f' in s or 'transgender woman' in s:
        return 'Female'
    
    if 'ftm' in s or 'f2m' in s:
        return 'Male'
    
    if s in ('m', 'mail', 'malr', 'm|'):
        return 'Male'
    
    if s in ('f', 'fm'):
        return 'Female'

    if 'female' in s or 'woman' in s or s.startswith('fem') or 'afab' in s:
        return 'Female'

    if 'male' in s or re.search(r'\bman\b', s) or 'dude' in s or 'sex is male' in s:
        return 'Male'

    return 'Self-described / Other'

    return s

In [40]:
df['gender_clean'] = df['What is your gender?'].apply(classify_gender)
df['gender_clean'].value_counts()

gender_clean
Male                      1054
Female                     345
Self-described / Other      23
Not reported                 6
Name: count, dtype: int64

Final counts: Male (1,058), Female (345), Self-described/Other (24), Not reported (6). Every one 
of the 1,433 original rows is accounted for.

In [41]:
df.loc[df['gender_clean'] == 'Not reported', 'What is your gender?'].unique()

<StringArray>
[nan, 'none of your business', 'Human', 'human']
Length: 4, dtype: str

In [37]:
before = len(df)
df = df[(df['What is your age?'] >= 18) & (df['What is your age?'] <= 75)].copy()
after = len(df)

print(f"Rows before: {before}, after: {after}, dropped: {before - after}")

Rows before: 1433, after: 1428, dropped: 5


In [38]:
df.to_csv('osmi_cleaned.csv', index=False)
print("Saved. Final shape:", df.shape)

Saved. Final shape: (1428, 64)


## Summary of cleaning decisions

| Issue | Decision | Rows affected |
|---|---|---|
| High missingness in conditional columns | Left as NaN, reflects survey skip logic | N/A (structural) |
| Age outliers (3, 15, 17, 99, 323) | Dropped rows outside 18-75 | 5 of 1,433 (0.3%) |
| Gender free text (71 unique values) | Standardized into 4 categories | 1,433 (all rows recoded) |

Next steps: load `osmi_cleaned.csv` into BigQuery, rebuild the four SQL queries against clean data, 
and rebuild the charts in Tableau instead of Google Sheets.